# Part 1c: Energy Performance Certificate audit (Oxfordshire, 2015–2025)

Audit of the domestic EPC extract: choosing 28 of 93 columns to keep, then checking floor areas, room counts, floor heights, nulls, duplicate certificates per property and dates.

Findings and decisions: `logs/part_1_structural_anomalies.md`

**Note on outputs:** the outputs of this notebook have been cleared because
address matching displays individual address-level records (addresses,
postcodes). The code and commentary are unchanged, and the results
(match rates, funnel figures and failure cases) are summarised in the
markdown cells below and in [`logs/`](../logs/). To see the outputs, run
the notebook locally with the source data (see the README).

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df_raw = pd.read_csv('../data/raw/EDC_OXFORDSHIRE_2015_2025.csv')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

In [ ]:
print(df_raw.columns)
print(df_raw.shape)

### Which columns to keep?

In [ ]:
df_raw.isna().sum()

In [ ]:
cols = ['local_authority', 'constituency', 'constituency_label', 'region', 'country']

print(df_raw[cols].nunique())
print(df_raw['local_authority'].value_counts())
print(df_raw['constituency'].value_counts())
print(df_raw['constituency_label'].value_counts())
print(df_raw['region'].value_counts())

- Country can be dropped (1 val)
- region can be dropped (only 4 in a second value)
- constituency shows parliamentary constituencies that cut accross boundaries (drop)
- local authority lines up with district in the price paid set
What are the 4 rows with a different region and constituency?:

In [ ]:
df_raw[df_raw['region'] == 'E12000005']

Drop them as local_authority falls outside the 5 target districts. They are on the Warwickshire border.

In [ ]:
dates = ['inspection_date', 'lodgement_date', 'lodgement_datetime']

print(df_raw[dates])
print(f"{(df_raw['inspection_date'] == df_raw['lodgement_date']).sum()/df_raw.shape[0]*100:.2f}% of inspections lodged on the same day")
print((df_raw['lodgement_date'].astype('datetime64[ns]') - df_raw['inspection_date'].astype('datetime64[ns]')).describe())

- lodgement_date = inspection_date 54% of time
- lodgement_datetime has time precision that is not needed for matching with price paid data
- time between inspection and lodgement is median 0 days, mean 8 days with min 0 days, max of 549 days and 75 percentile 2 days. So some very long delays are distorting the average, none are lodged before their inspections.

In [ ]:
energy = ['current_energy_rating', 'current_energy_efficiency',
    'co2_emissions_current',
    'co2_emissions_potential',
    'energy_consumption_current',
    'energy_consumption_potential',
    'environment_impact_current',
    'environment_impact_potential',
    'heating_cost_current',
    'heating_cost_potential',
    'hot_water_cost_current',
    'hot_water_cost_potential',
    'lighting_cost_current',
    'lighting_cost_potential',
]

energy_current = ['current_energy_rating', 'current_energy_efficiency',
    'co2_emissions_current',
    'energy_consumption_current',
    'environment_impact_current',
    'heating_cost_current',
    'hot_water_cost_current',
    'lighting_cost_current',
]

print(df_raw[energy].corr(numeric_only=True))

sns.heatmap(abs(df_raw[energy_current].corr(numeric_only=True)))

- drop 'potential' columns except for 'potential_energy_efficiency' to still give an optional measure of room of improvement
- do costs correlate strongly with floor area?

In [ ]:
energy_current_floor = ['total_floor_area', 'current_energy_rating', 'current_energy_efficiency',
    'co2_emissions_current',
    'energy_consumption_current',
    'environment_impact_current',
    'heating_cost_current',
    'hot_water_cost_current',
    'lighting_cost_current',
]
print(df_raw[energy_current_floor].corr(numeric_only=True))

- Keep heating_cost_current as its an intuitive running cost for a buyer
- drop co2_emissions_current (correlates 0.82 with heating), drop lighting_cost_current (small and correlated 0.66 with floor area), drop hot_water_cost_current (small)
- keep current_energy_efficiency and energy_consumption_current (size-independant)

In [ ]:
df_raw['transaction_type'].value_counts(normalize=True)

Useful information for matching with price database

In [ ]:
df_raw['floor_height'].value_counts()

Shows a variety of values, keep

In [ ]:
df_raw.columns

In [ ]:
full_column_list = ['certificate_number', 'address1', 'address2', 'address3', 'postcode',
       'posttown', 'address', 'constituency', 'constituency_label',
       'local_authority', 'local_authority_label', 'built_form',
       'co2_emiss_curr_per_floor_area', 'co2_emissions_current',
       'co2_emissions_potential', 'construction_age_band',
       'current_energy_efficiency', 'current_energy_rating',
       'energy_consumption_current', 'energy_consumption_potential',
       'energy_tariff', 'environment_impact_current',
       'environment_impact_potential', 'extension_count',
       'fixed_lighting_outlets_count', 'flat_storey_count', 'flat_top_storey',
       'floor_description', 'floor_energy_eff', 'floor_height', 'floor_level',
       'glazed_area', 'glazed_type', 'heat_loss_corridor',
       'heating_cost_current', 'heating_cost_potential',
       'hot_water_cost_current', 'hot_water_cost_potential',
       'hot_water_energy_eff', 'hot_water_env_eff', 'hotwater_description',
       'inspection_date', 'lighting_cost_current', 'lighting_cost_potential',
       'lighting_description', 'lighting_energy_eff', 'lighting_env_eff',
       'lodgement_date', 'lodgement_datetime', 'low_energy_lighting',
       'low_energy_fixed_lighting_outlets_count', 'main_fuel',
       'mainheat_description', 'mainheat_energy_eff', 'mainheat_env_eff',
       'mainheatc_energy_eff', 'mainheatc_env_eff', 'mainheatcont_description',
       'main_heating_controls', 'mains_gas_flag', 'mechanical_ventilation',
       'multi_glaze_proportion', 'number_habitable_rooms',
       'number_heated_rooms', 'number_open_fireplaces', 'photo_supply',
       'potential_energy_efficiency', 'potential_energy_rating',
       'property_type', 'report_type', 'roof_description', 'roof_energy_eff',
       'roof_env_eff', 'secondheat_description', 'sheating_energy_eff',
       'sheating_env_eff', 'solar_water_heating_flag', 'tenure',
       'total_floor_area', 'transaction_type', 'unheated_corridor_length',
       'walls_description', 'walls_energy_eff', 'walls_env_eff',
       'wind_turbine_count', 'windows_description', 'windows_energy_eff',
       'windows_env_eff', 'floor_env_eff', 'region', 'country', 'uprn',
       'uprn_source']

columns_to_keep = [
    'certificate_number', 'address1', 'address2', 'address3', 'postcode',
    'posttown', 'address', 'local_authority', 'local_authority_label',
    'built_form', 'construction_age_band', 'current_energy_efficiency',
    'current_energy_rating', 'energy_consumption_current', 'extension_count',
    'flat_storey_count', 'floor_height', 'heating_cost_current',
    'inspection_date', 'lodgement_date', 'number_habitable_rooms',
    'number_heated_rooms', 'potential_energy_efficiency', 'property_type',
    'tenure', 'total_floor_area', 'transaction_type', 'uprn', 'uprn_source'
]

columns_to_drop = [
    'constituency', 'constituency_label', 'co2_emiss_curr_per_floor_area',
    'co2_emissions_current', 'co2_emissions_potential',
    'energy_consumption_potential', 'energy_tariff',
    'environment_impact_current', 'environment_impact_potential',
    'fixed_lighting_outlets_count', 'flat_top_storey', 'floor_description',
    'floor_energy_eff', 'floor_level', 'glazed_area', 'glazed_type',
    'heat_loss_corridor', 'heating_cost_potential', 'hot_water_cost_current',
    'hot_water_cost_potential', 'hot_water_energy_eff', 'hot_water_env_eff',
    'hotwater_description', 'lighting_cost_current', 'lighting_cost_potential',
    'lighting_description', 'lighting_energy_eff', 'lighting_env_eff',
    'lodgement_datetime', 'low_energy_lighting',
    'low_energy_fixed_lighting_outlets_count', 'main_fuel',
    'mainheat_description', 'mainheat_energy_eff', 'mainheat_env_eff',
    'mainheatc_energy_eff', 'mainheatc_env_eff', 'mainheatcont_description',
    'main_heating_controls', 'mains_gas_flag', 'mechanical_ventilation',
    'multi_glaze_proportion', 'number_open_fireplaces', 'photo_supply',
    'potential_energy_rating', 'report_type', 'roof_description',
    'roof_energy_eff', 'roof_env_eff', 'secondheat_description',
    'sheating_energy_eff', 'sheating_env_eff', 'solar_water_heating_flag',
    'unheated_corridor_length', 'walls_description', 'walls_energy_eff',
    'walls_env_eff', 'wind_turbine_count', 'windows_description',
    'windows_energy_eff', 'windows_env_eff', 'floor_env_eff', 'region',
    'country'
]

full_set = set(full_column_list)
keep_set = set(columns_to_keep)
drop_set = set(columns_to_drop)


print("Duplicates in full list:", len(full_column_list) - len(full_set))
print("Duplicates in columns_to_keep:", len(columns_to_keep) - len(keep_set))
print("Duplicates in columns_to_drop:", len(columns_to_drop) - len(drop_set))

overlap = keep_set.intersection(drop_set)
print("Columns in BOTH keep & drop:", overlap)

missing = full_set - (keep_set | drop_set)
print("Columns missing from both lists:", missing)

extra = (keep_set | drop_set) - full_set
print("Unexpected columns (not in full list):", extra)

In [ ]:
df = df_raw[columns_to_keep].copy()

df.head()

df[['number_habitable_rooms', 'number_heated_rooms']].corr()
(df.shape[0] - (df['number_habitable_rooms'] == df['number_heated_rooms']).sum()) / df.shape[0]

Is heated rooms redundant with habitable rooms?
No there are 25% of cases where they are not equal

In [ ]:
(df['number_heated_rooms'] > df['number_habitable_rooms']).sum()

Heated rooms only exceeds habitable in one case

### Data audit on kept columns

Drop 4 rows with out-of-scope local_authority

In [ ]:
df = df[df['local_authority'] != 'E07000221']

df['local_authority'].value_counts()

In [ ]:
df.dtypes

#### Core numeric features

In [ ]:
print(df['total_floor_area'].describe())

print(f"There are {(df['total_floor_area'] < 10).sum()} records with < 10 sqm floor areas and {(df['total_floor_area'] > 2000).sum()} with > 2000 sqm")

print(df[['address1', 'number_habitable_rooms', 'total_floor_area', 'property_type']][df['total_floor_area'] < 10].sort_values(by = 'number_habitable_rooms', ascending = False))

In [ ]:
df['area_per_hab_room'] = df['total_floor_area'] / df['number_habitable_rooms']
print(df[['certificate_number', 'address1', 'number_habitable_rooms', 'total_floor_area', 'area_per_hab_room', 'property_type']][df['total_floor_area'] > 2000].sort_values(by = 'number_habitable_rooms', ascending = False))

print(df['area_per_hab_room'].describe())

print(f"\nThere are {df['total_floor_area'].isna().sum()} null values")

In [ ]:
sns.histplot(df['total_floor_area'])

- total_floor_area
    - mean = 97, median = 83, min = 3, max = 5133, 75% = 111
    - There are 27 records with < 10 sqm floor areas and 8 with > 2000 sqm
        - 19 records are flats in the same building (7 or 9 sqm) with 1 habitable room
        - 1 is a room of 3 sqm
        - 7 are records with 2-4 habitable rooms but 5-9 sqm floor area (each room <= 3 sqm)
            - 3 are house or maisonette, size is too small to be real, exclude
            - 4 are flats.
        - National space standards put a bedroom at 6.5-7.5sqm. *Proposed: Exclude these 7 records with multiple rooms and < 9 sqm. Exclude the record with 3 sqm. Keep the 19 records that are 1 room flats (7-9 sqm).*
    - There are 8 records with > 2000 sqm
        - 2 have NaN habitable rooms
        - 5 have 3 - 7 habitable rooms (implausible)
        - 1 has 39 habitable rooms (plausible)
        - perhaps average floor area per habitable room would be a good measure. These have up to 1209 sqm per room.
            - On full dataset it ranges from 0.8 sqm/room to 1209 sqm/room, median 20 sqm, mean 21 sqm.
        - *Proposed: Exclude the 7 records with >2000 sqm and <= 7 rooms (8206-7322-4290-4245-7906, 0552-3041-5201-3644-6204, 1032-5920-2209-0815-0202, 0044-2876-6518-9608-3961, 8171-6324-5580-7887-6996, 3434-1132-3000-0328-7226, 5135-0635-7000-0689-9206)*
    - Small areas will have a larger effect on £ per sqm calculations later, so important to clean.

        

In [ ]:
col = ['number_habitable_rooms', 'number_heated_rooms', 'extension_count', 'flat_storey_count', 'floor_height']
for i in col:
    print(f"The range of values for {i} is {df[i].min()} to {df[i].max()}, with median of {df[i].median()}")
    print(f"There are {df[i].isna().sum()} null values\n")


print((df['floor_height'] > 5).sum())
print(df[df['floor_height'] > 5].sort_values(by='floor_height', ascending=False).head(10))

In [ ]:

print(f"There are {df[df['number_heated_rooms'] == 0].shape[0]} records with 0 heated rooms")
print(f"There are {df[df['floor_height'] < 1.5].shape[0]} records with floor height < 1.5m. Including {df[df['floor_height'] < 1].shape[0]} records with floor height < 1m.")

In [ ]:
print(pd.concat(
    [df['transaction_type'].value_counts(normalize=True), 
    df[df['number_habitable_rooms'].isna()]['transaction_type'].value_counts(normalize=True)], axis=1, keys=['full_set_%', 'null_rooms_%']))

print(df[df['number_habitable_rooms'].isna()].head())

In [ ]:
print(f"Flat storey count is null:")
print(pd.concat([df['property_type'].value_counts(normalize=True), df[df['flat_storey_count'].isna()]['property_type'].value_counts(normalize=True)], axis=1, keys=['full_set_%', 'null_set_%']))


print(f"\nFloor height is null:")
print(pd.concat([df['property_type'].value_counts(normalize=True), df[df['floor_height'].isna()]['property_type'].value_counts(normalize=True)], axis=1, keys=['full_set_%', 'null_set_%']))

In [ ]:
print(f"Percentage of New dwelling records with null 'number_heated_rooms': {100 * (df[df['transaction_type'] == 'New dwelling']['number_heated_rooms'].isna()).sum() / df[df['transaction_type'] == 'New dwelling'].shape[0]}%")

In [ ]:
print(df.groupby('property_type')['flat_storey_count'].agg('median'))
print(df[df['flat_storey_count'] > 3]['property_type'].value_counts())
print(df[df['flat_storey_count'] > 4]['property_type'].value_counts())

- 'number_habitable_rooms', 'number_heated_rooms', 'extension_count', 'flat_storey_count', 'floor_height'
    - The range of values for number_habitable_rooms is 1.0 to 81.0, with median of 4.0
    - The range of values for number_heated_rooms is 0.0 to 55.0, with median of 4.0
          - Max room counts will need floor size to audit (or my sqm/room calculation), but 81 or 55 rooms is high
          - Zero heated rooms is implausible (338 records). *Proposed: Treat records with 0 heated rooms as value is missing.*
    - The range of values for 'extension_count' is 0.0 to 4.0, with median of 0.0
        - Reasonable
    - The range of values for 'flat_storey_count' is 1.0 to 9.0, with median of 2.0
        - 430 records with 'flat_storey_count' > 3 (428 Houses). 11 records with 'flat_storey_count' > 4 (11 Houses, up to 9 storeys). 
        - Reading online suggests this should be a measure of the size of a block of flats, havent found a good source for column definitions. *Proposed: Definition unclear, not a reliable feature unless clarified (drop)*
    - The range of values for 'floor_height' is 0.0 to 8.87, with median of 2.38
        - 53 records with 'floor height' > 5m. Many are built <1900. Believable for say converted churches or houses with mezzanine?
        - There are 2516 records with 'floor height' < 1.5m. Including 98 records with 'floor height' < 1m. *Proposed: Treat records with floor height < 1.5m as missing.*
    - Nulls
        - There are 49419 records with null for all of 'number_habitable_rooms', 'number_heated_rooms', 'extension_count'
        - These are disproportionately new dwellings (91% vs 21% new dwellings in wider sample)
        - All New dwelling records ('transaction_type') have null 'number_heated_rooms'. This is consistent with a different type of assessment for new builds.
        - 'flat_storey_count' is null in 5016 cases
            - Distribution of 'property_type' (e.g. flat vs house etc) slightly skewed toward flats (21 vs 32%)
        - 'floor_height'
            - 1615 null values. 99% are for Flats where the 'floor_height' is likely to be very standardised. *Proposed: Fill flat nulls with the average flat floor_height.*
  

Duplicate certificates

In [ ]:
print(df.shape[0] - df['certificate_number'].nunique())
print(df['certificate_number'].isna().sum())
print(df.duplicated(subset='certificate_number').sum())

print(df.shape[0] - df['uprn'].nunique())
print(df['uprn'].isna().sum())
print(df.groupby('uprn')['certificate_number'].agg('count').sort_values(ascending=False))
print(df.duplicated(subset='uprn').sum())

- 'certificate_number' is unique with no missing values. None flagged as duplicated
- each 'uprn' has up to 50 'certificate_number' associated with it. 26776 flagged as duplicated
- 

In [ ]:
categorical = ['property_type', 'built_form', 'construction_age_band', 'tenure', 'transaction_type', 'current_energy_rating', 'uprn_source']

for i in categorical:
    print(f"{i} has {df[i].nunique()} unique values.")
    print(df[i].value_counts())

In [ ]:
df.groupby('construction_age_band')['certificate_number'].count().sort_values(ascending=False).head(50)

| Column | Unique values | Top values | Placeholders / anomalies | Proposed treatment |
|---|---|---|---|---|
| `property_type` | 5|House, Flat |'Not Recorded' (1305, 0.6%) |None |
| `built_form` | 7|Semi-Detached, Detached | | |
| `construction_age_band` | 67|England and Wales: 1950-1966, England and Wales: 1967-1975 |Mixture of bands and specific years. 201 as year|Convert single years into bands and exclude 201 |
| `tenure` |4 |owner-occupied, rented (private) | | |
| `transaction_type` |14 |Marketed sale, Rental  |Contains 'None of the above' |Likely none needed as less common types may not correspond to a sale |
| `current_energy_rating` | 7|C, D, B | | |
| `uprn_source` |2 |Energy Assessor, Address Matched  | | |

In [ ]:
print(df['inspection_date'].describe())
df['lodgement_date'] = df['lodgement_date'].astype('datetime64[ns]')
print(df['lodgement_date'].describe())

In [ ]:
df['inspection_date'] = df['inspection_date'].astype('datetime64[ns]') 
print((df['inspection_date'] < '2015-01-01').sum())
print(df[df['inspection_date'] < '2015-01-01'].head())
print((df['inspection_date'] > '2025-12-31').sum())
print(df['lodgement_date'].isna().sum())
print(df['inspection_date'].isna().sum())

- 'inspection_date' & 'lodgement_date'
    - format: YYYY-MM-DD
    - 'lodgement_date' fully within intended 2015-01-01 - 2015-12-31 limits
    - 468 records where the 'inspection_date' falls in 2014 but 'lodgement_date' is in 2015
    - No 'inspection_date' after 2025-12-31
    - No 'inspection_date' or 'lodgement_date' records are null

In [ ]:
from pathlib import Path
folder = Path('../data/raw')
file_gen = folder.glob('ppd_data*.csv')
pp = pd.concat([pd.read_csv(f) for f in file_gen])

In [ ]:
pp_postcode = set(pp['postcode'])
epc_postcode = set(df['postcode'])

matched = 100* len(epc_postcode.intersection(pp_postcode)) / len(pp_postcode)
print(f"{matched:.1f}")

- 'postcode' matching pre-check
    - Set of 'postcode' compared with price paid 'postcode' set. Found that 94.7% of the price paid postcodes are found in the EPC database.

- Can confirm 'address' is a combination of 'address1', 'address2' and 'address3'

In [ ]:
df[['current_energy_rating', 'current_energy_efficiency']].head()

bins = [0, 20, 38, 54, 68, 80, 91, 200]
bands = ['G', 'F', 'E', 'D', 'C', 'B', 'A']
df['calc_band'] = pd.cut(df['current_energy_efficiency'], bins=bins, labels=bands, include_lowest=True)

print(pd.concat([df['current_energy_rating'].value_counts(), df['calc_band'].value_counts()], axis=1))
print(f" There is a {100 * (df['current_energy_rating'] == df['calc_band']).sum() / df.shape[0]:.2f}% match between given and calculated energy_rating")

print(df['current_energy_efficiency'].describe())
print(((df['current_energy_efficiency'] > 110) & (df['energy_consumption_current'] > 0)).sum())
print(df[(df['current_energy_efficiency'] > 110) & (df['energy_consumption_current'] > 0)])

- Checking 'current_energy_rating' against 'current_energy_efficiency'
    - Calculated energy ratings from efficiency. Perfect agreement between.
    - Worth noting that the efficiency can go over 100 if a property generates more energy (ultra-modern properties with renewable energy generation) than it uses. There are 417 records with efficiency > 100 inc 41 with efficiency > 110.
    - There are 6 records with an efficiencies over 110 but with positive 'energy_consumption_current'. These are edge cases, unusual but plausible as they may still use a large amount of energy in winter months. Two have energy_consumption_current >200 and are Maisonette/Flat which flags them as likely errors. *Proposed: Exclude 2051-7735-3040-1307-9975 and 0061-1212-2504-8687-0900*